# Periodic NaCl Forward Equivariance Check

This notebook keeps the check close to the OA-ReactDiff demo style:

1. run the model on the original periodic joint graph
2. rotate the input graph
3. run the model again
4. directly compare forward outputs: `h - h_rot` and `pos @ R - pos_rot`


In [1]:
from pathlib import Path
import sys

import torch
from ase.io import read, write
from torch.utils.data import DataLoader
from e3nn import o3

repo_root = Path.cwd()
# 如果当前目录下没有dataset文件夹，但父目录下有，则将repo_root设置为父目录
if not (repo_root / 'dataset').exists() and (repo_root.parent / 'dataset').exists():
    repo_root = repo_root.parent
print(repo_root)
# 加入父目录到sys.path，以便导入akmcgc模块
if str(repo_root.parent) not in sys.path:
    sys.path.insert(0, str(repo_root.parent))

from akmcgc.dataset import RxnDataset # Dataset for reaction data
from akmcgc.model import EGNN, LEFTNet # GNN models

torch.manual_seed(2024) # 设置随机种子以确保结果可复现
data_dir = repo_root / 'tests' / 'data'


/Users/wx/Desktop/yyxwjq/akmcgc


## Debug Contract Helpers

这些 helper 用来统一打印模型输入、输出和参数矩阵维度。


In [2]:
def tensor_contract(name, value):
    if torch.is_tensor(value):
        return f"{name:16s} shape={tuple(value.shape)!s:18s} dtype={str(value.dtype):14s} device={value.device}"
    return f"{name:16s} value={value!r}"


def print_tensor_contract(title, tensors):
    print(f"\n[{title}]")
    for name, value in tensors.items():
        print(tensor_contract(name, value))


def print_parameter_contract(title, module, max_rows=30):
    print(f"\n[{title} parameter contract]")
    total = 0
    trainable = 0
    rows = []
    for name, param in module.named_parameters():
        n = param.numel()
        total += n
        if param.requires_grad:
            trainable += n
        rows.append((name, tuple(param.shape), param.dtype, param.requires_grad, n))
    print(f"total parameters={total:,}, trainable={trainable:,}, tensors={len(rows)}")
    for name, shape, dtype, requires_grad, n in rows[:max_rows]:
        print(f"{name:48s} shape={shape!s:18s} dtype={str(dtype):14s} trainable={requires_grad!s:5s} n={n:,}")
    if len(rows) > max_rows:
        print(f"... {len(rows) - max_rows} more parameter tensors")


## Build A Batched Periodic Joint Graph

In [3]:
react_single = read(data_dir / 'nacl_crystal_react.extxyz')
product_single = read(data_dir / 'nacl_crystal_product.extxyz')

react_repeat = data_dir / 'nacl_repeat20_react.extxyz'
product_repeat = data_dir / 'nacl_repeat20_product.extxyz'
write(react_repeat, [react_single.copy() for _ in range(20)])
write(product_repeat, [product_single.copy() for _ in range(20)])

dataset = RxnDataset(
    react_file=str(react_repeat),
    product_file=str(product_repeat),
    cutoff=4.5,
    max_neigh=32,
    r_fixed=True,
    r_pbc=True,
    device='cpu',
)
loader = DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=RxnDataset.collate_fn)
batch = next(iter(loader))

print('h', tuple(batch['h'].shape))
print('pos', tuple(batch['pos'].shape))
print('edge_index', tuple(batch['edge_index'].shape))
print('cell_offsets', tuple(batch['cell_offsets'].shape))
print('cell', tuple(batch['cell'].shape))
print('pbc', tuple(batch['pbc'].shape))

assert torch.allclose(batch['h'][:, :3], batch['pos'])
assert torch.all(batch['fragment'][batch['edge_index'][0]] == batch['fragment'][batch['edge_index'][1]])


h (128, 122)
pos (128, 3)
edge_index (2, 2304)
cell_offsets (2304, 3)
cell (4, 2, 3, 3)
pbc (4, 2, 3)


## Build EGNN And LEFTNet

In [4]:
feature_dim = batch['h'].shape[1] - 3

egnn = EGNN(
    in_node_nf=feature_dim,
    in_edge_nf=0,
    hidden_nf=128,
    edge_hidden_nf=128,
    n_layers=3,
    attention=True,
    out_node_nf=feature_dim,
    tanh=True,
    coords_range=10.0,
    norm_constant=1.0,
    inv_sublayers=3,
    sin_embedding=False,
    normalization_factor=1.0,
    aggregation_method='mean',
    reflect_equiv=True,
).to(dtype=batch['pos'].dtype).eval()

leftnet = LEFTNet(
    pos_require_grad=False,
    cutoff=4.5,
    num_layers=2,
    hidden_channels=64,
    num_radial=32,
    in_hidden_channels=feature_dim,
    reflect_equiv=True,
    legacy=True,
    update=True,
    pos_grad=False,
    single_layer_output=True,
    object_aware=True,
).to(dtype=batch['pos'].dtype).eval()


## Model Parameter Contracts

这里明确列出 EGNN / LEFTNet 的参数矩阵。`next(model.parameters())` 只会拿到这里的第一个参数矩阵，不代表只有一个参数。


In [5]:
print('feature_dim =', feature_dim)
print_parameter_contract('EGNN', egnn, max_rows=24)
print_parameter_contract('LEFTNet', leftnet, max_rows=24)

for name, model in [('EGNN', egnn), ('LEFTNet', leftnet)]:
    print_tensor_contract(f'{name} forward inputs', {
        'node_h': batch['h'][:, 3:],
        'pos': batch['pos'],
        'edge_index': batch['edge_index'],
        'cell': batch['cell'],
        'pbc': batch['pbc'],
        'cell_offsets': batch['cell_offsets'],
        'neighbors': batch['neighbors'],
        'fragment': batch['fragment'],
        'mask': batch['mask'],
    })
    with torch.no_grad():
        h_out, pos_out, edge_attr_out = model(
            batch['h'][:, 3:], batch['pos'], batch['edge_index'], edge_attr=None,
            cell=batch['cell'], pbc=batch['pbc'], cell_offsets=batch['cell_offsets'],
            neighbors=batch['neighbors'], fragment=batch['fragment'], mask=batch['mask'],
        )
    print_tensor_contract(f'{name} forward outputs', {
        'h_out': h_out,
        'pos_out': pos_out,
        'edge_attr_out': edge_attr_out,
    })


feature_dim = 119

[EGNN parameter contract]
total parameters=1,273,441, trainable=1,273,441, tensors=128
embedding.weight                                 shape=(128, 119)         dtype=torch.float64  trainable=True  n=15,232
embedding.bias                                   shape=(128,)             dtype=torch.float64  trainable=True  n=128
embedding_out.weight                             shape=(119, 128)         dtype=torch.float64  trainable=True  n=15,232
embedding_out.bias                               shape=(119,)             dtype=torch.float64  trainable=True  n=119
edge_embedding.weight                            shape=(127, 1)           dtype=torch.float64  trainable=True  n=127
edge_embedding.bias                              shape=(127,)             dtype=torch.float64  trainable=True  n=127
edge_embedding_out.weight                        shape=(1, 127)           dtype=torch.float64  trainable=True  n=127
edge_embedding_out.bias                          shape=(1,)          

## EGNN: Global Rotation

For a global periodic rotation, rotate both `pos` and `cell`. Keep `cell_offsets` unchanged because they are integer lattice offsets.

In [6]:
model = egnn
torch.manual_seed(0)
rot = o3.rand_matrix().to(dtype=batch['pos'].dtype, device=batch['pos'].device)

with torch.no_grad():
    h, pos, _ = model.forward(
        batch['h'][:, 3:],
        batch['pos'],
        batch['edge_index'],
        edge_attr=None,
        cell=batch['cell'],
        pbc=batch['pbc'],
        cell_offsets=batch['cell_offsets'],
        neighbors=batch['neighbors'],
        fragment=batch['fragment'],
        mask=batch['mask'],
    )

batch_rot = {k: v.clone() if torch.is_tensor(v) else v for k, v in batch.items()}
batch_rot['pos'] = batch['pos'] @ rot
batch_rot['cell'] = batch['cell'] @ rot
batch_rot['h'][:, :3] = batch_rot['pos']

with torch.no_grad():
    h_rot, pos_rot, _ = model.forward(
        batch_rot['h'][:, 3:],
        batch_rot['pos'],
        batch_rot['edge_index'],
        edge_attr=None,
        cell=batch_rot['cell'],
        pbc=batch_rot['pbc'],
        cell_offsets=batch_rot['cell_offsets'],
        neighbors=batch_rot['neighbors'],
        fragment=batch_rot['fragment'],
        mask=batch_rot['mask'],
    )

egnn_h_global = torch.max(torch.abs(h - h_rot))
egnn_pos_global = torch.max(torch.abs(pos @ rot - pos_rot))

print('EGNN max |h - h_rot| =', egnn_h_global.item())
print('EGNN max |pos @ rot - pos_rot| =', egnn_pos_global.item())


EGNN max |h - h_rot| = 2.6461282054413005e-09
EGNN max |pos @ rot - pos_rot| = 2.5653978941164723e-11


## EGNN: Rotate Only Fragment 0

In [7]:
model = egnn
frag = 0
frag_mask = batch['fragment'] == frag
torch.manual_seed(0)
rot = o3.rand_matrix().to(dtype=batch['pos'].dtype, device=batch['pos'].device)

with torch.no_grad():
    h, pos, _ = model.forward(
        batch['h'][:, 3:],
        batch['pos'],
        batch['edge_index'],
        edge_attr=None,
        cell=batch['cell'],
        pbc=batch['pbc'],
        cell_offsets=batch['cell_offsets'],
        neighbors=batch['neighbors'],
        fragment=batch['fragment'],
        mask=batch['mask'],
    )

batch_frag_rot = {k: v.clone() if torch.is_tensor(v) else v for k, v in batch.items()}
batch_frag_rot['pos'][frag_mask] = batch_frag_rot['pos'][frag_mask] @ rot
batch_frag_rot['cell'][:, frag] = batch_frag_rot['cell'][:, frag] @ rot
batch_frag_rot['h'][:, :3] = batch_frag_rot['pos']

with torch.no_grad():
    h_frag_rot, pos_frag_rot, _ = model.forward(
        batch_frag_rot['h'][:, 3:],
        batch_frag_rot['pos'],
        batch_frag_rot['edge_index'],
        edge_attr=None,
        cell=batch_frag_rot['cell'],
        pbc=batch_frag_rot['pbc'],
        cell_offsets=batch_frag_rot['cell_offsets'],
        neighbors=batch_frag_rot['neighbors'],
        fragment=batch_frag_rot['fragment'],
        mask=batch_frag_rot['mask'],
    )

pos_expected = pos.clone()
pos_expected[frag_mask] = pos_expected[frag_mask] @ rot

egnn_h_frag = torch.max(torch.abs(h - h_frag_rot))
egnn_pos_frag = torch.max(torch.abs(pos_expected - pos_frag_rot))

print('EGNN max |h - h_frag_rot| =', egnn_h_frag.item())
print('EGNN max |expected_pos - pos_frag_rot| =', egnn_pos_frag.item())


EGNN max |h - h_frag_rot| = 2.537404286684364e-09
EGNN max |expected_pos - pos_frag_rot| = 6.938893903907228e-18


## LEFTNet: Global Rotation

In [8]:
model = leftnet
torch.manual_seed(0)
rot = o3.rand_matrix().to(dtype=batch['pos'].dtype, device=batch['pos'].device)

with torch.no_grad():
    h, pos, _ = model.forward(
        batch['h'][:, 3:],
        batch['pos'],
        batch['edge_index'],
        edge_attr=None,
        cell=batch['cell'],
        pbc=batch['pbc'],
        cell_offsets=batch['cell_offsets'],
        neighbors=batch['neighbors'],
        fragment=batch['fragment'],
        mask=batch['mask'],
    )

batch_rot = {k: v.clone() if torch.is_tensor(v) else v for k, v in batch.items()}
batch_rot['pos'] = batch['pos'] @ rot
batch_rot['cell'] = batch['cell'] @ rot
batch_rot['h'][:, :3] = batch_rot['pos']

with torch.no_grad():
    h_rot, pos_rot, _ = model.forward(
        batch_rot['h'][:, 3:],
        batch_rot['pos'],
        batch_rot['edge_index'],
        edge_attr=None,
        cell=batch_rot['cell'],
        pbc=batch_rot['pbc'],
        cell_offsets=batch_rot['cell_offsets'],
        neighbors=batch_rot['neighbors'],
        fragment=batch_rot['fragment'],
        mask=batch_rot['mask'],
    )

leftnet_h_global = torch.max(torch.abs(h - h_rot))
leftnet_pos_global = torch.max(torch.abs(pos @ rot - pos_rot))

print('LEFTNet max |h - h_rot| =', leftnet_h_global.item())
print('LEFTNet max |pos @ rot - pos_rot| =', leftnet_pos_global.item())


LEFTNet max |h - h_rot| = 7.345086885934826e-08
LEFTNet max |pos @ rot - pos_rot| = 1.3587913016976927e-09


## LEFTNet: Rotate Only Fragment 0

In [9]:
model = leftnet
frag = 0
frag_mask = batch['fragment'] == frag
torch.manual_seed(0)
rot = o3.rand_matrix().to(dtype=batch['pos'].dtype, device=batch['pos'].device)

with torch.no_grad():
    h, pos, _ = model.forward(
        batch['h'][:, 3:],
        batch['pos'],
        batch['edge_index'],
        edge_attr=None,
        cell=batch['cell'],
        pbc=batch['pbc'],
        cell_offsets=batch['cell_offsets'],
        neighbors=batch['neighbors'],
        fragment=batch['fragment'],
        mask=batch['mask'],
    )

batch_frag_rot = {k: v.clone() if torch.is_tensor(v) else v for k, v in batch.items()}
batch_frag_rot['pos'][frag_mask] = batch_frag_rot['pos'][frag_mask] @ rot
batch_frag_rot['cell'][:, frag] = batch_frag_rot['cell'][:, frag] @ rot
batch_frag_rot['h'][:, :3] = batch_frag_rot['pos']

with torch.no_grad():
    h_frag_rot, pos_frag_rot, _ = model.forward(
        batch_frag_rot['h'][:, 3:],
        batch_frag_rot['pos'],
        batch_frag_rot['edge_index'],
        edge_attr=None,
        cell=batch_frag_rot['cell'],
        pbc=batch_frag_rot['pbc'],
        cell_offsets=batch_frag_rot['cell_offsets'],
        neighbors=batch_frag_rot['neighbors'],
        fragment=batch_frag_rot['fragment'],
        mask=batch_frag_rot['mask'],
    )

pos_expected = pos.clone()
pos_expected[frag_mask] = pos_expected[frag_mask] @ rot

leftnet_h_frag = torch.max(torch.abs(h - h_frag_rot))
leftnet_pos_frag = torch.max(torch.abs(pos_expected - pos_frag_rot))

print('LEFTNet max |h - h_frag_rot| =', leftnet_h_frag.item())
print('LEFTNet max |expected_pos - pos_frag_rot| =', leftnet_pos_frag.item())


LEFTNet max |h - h_frag_rot| = 7.345086885934826e-08
LEFTNet max |expected_pos - pos_frag_rot| = 1.2016556638627662e-09


## Simple Pass/Fail Threshold

In [10]:
atol = 2e-3

assert egnn_h_global < atol
assert egnn_pos_global < atol
assert egnn_h_frag < atol
assert egnn_pos_frag < atol

assert leftnet_h_global < atol
assert leftnet_pos_global < atol
assert leftnet_h_frag < atol
assert leftnet_pos_frag < atol

print('All forward-output equivariance checks passed.')


All forward-output equivariance checks passed.
